# Notebook 02 — SimCLR Self-Supervised Pre-training

This notebook:
1. Runs SimCLR contrastive pre-training on PlantVillage (label-free)
2. Plots the NT-Xent loss curve
3. Visualises learned representations using t-SNE

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

from src.simclr import pretrain
from src.model import SimCLRModel
from src.dataset import build_simclr_dataloader
from src.utils import get_device, load_checkpoint, set_seed

set_seed(42)
DEVICE = get_device()
print(f'Using device: {DEVICE}')

DATA_ROOT      = '../data/PlantVillage'        # <- change to your path
CHECKPOINT_DIR = '../checkpoints'

## 1. Run SimCLR Pre-training

⚠️ This can take several hours on CPU. On a GPU it takes ~2-3 hours for 100 epochs.
Reduce `num_epochs` to 5-10 for a quick test run.

In [ ]:
ckpt_path = pretrain(
    data_root      = DATA_ROOT,
    checkpoint_dir = CHECKPOINT_DIR,
    num_epochs     = 100,         # Reduce to 5 for quick test
    batch_size     = 128,
    lr             = 3e-4,
    temperature    = 0.5,
    num_workers    = 4,
    device         = DEVICE,
)
print(f'Encoder saved: {ckpt_path}')

## 2. Plot NT-Xent Loss Curve

In [ ]:
ckpt = torch.load(ckpt_path, map_location='cpu')
history = ckpt['history']
epochs = range(1, len(history) + 1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs, history, color='#00d2ff', linewidth=2, label='NT-Xent Loss')

# Smoothed trend
window = max(1, len(history) // 10)
smoothed = np.convolve(history, np.ones(window)/window, mode='valid')
ax.plot(range(window, len(history)+1), smoothed, color='#e94560', linewidth=2,
        linestyle='--', label='Smoothed Trend')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('NT-Xent Loss', fontsize=12)
ax.set_title('SimCLR Pre-training Loss Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/simclr_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best loss: {ckpt["best_loss"]:.4f} (epoch {ckpt["epoch"]})')

## 3. t-SNE Visualisation of Learned Representations

Extracts 512 random leaf images, encodes them with the pre-trained encoder,
then visualises with t-SNE to see if disease classes cluster naturally.

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from src.dataset import get_val_transforms
import random

# Load model and encoder weights
model = SimCLRModel(pretrained_cnn=False).to(DEVICE)
ckpt  = torch.load(ckpt_path, map_location=DEVICE)
model.encoder.load_state_dict(ckpt['model_state'])
model.eval()

# Sample 512 images
ds = datasets.ImageFolder(DATA_ROOT, transform=get_val_transforms())
sample_idx = random.sample(range(len(ds)), min(512, len(ds)))
subset = Subset(ds, sample_idx)
loader = DataLoader(subset, batch_size=64, shuffle=False, num_workers=2)

# Extract embeddings
embeddings, labels_list = [], []
with torch.no_grad():
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        emb  = model.encoder.forward_features(imgs).cpu().numpy()
        embeddings.append(emb)
        labels_list.extend(lbls.numpy())

embeddings = np.vstack(embeddings)
labels_arr = np.array(labels_list)

print(f'Embedding matrix: {embeddings.shape}')
print('Running t-SNE (this may take a minute)...')

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
coords = tsne.fit_transform(embeddings)
print('t-SNE complete!')

In [ ]:
# Colour by healthy vs diseased
from src.utils import CLASS_NAMES

is_healthy = np.array(['healthy' in ds.classes[l] for l in labels_arr])

fig, ax = plt.subplots(figsize=(12, 9))
ax.scatter(coords[~is_healthy, 0], coords[~is_healthy, 1],
           c='#e94560', alpha=0.6, s=20, label='Diseased', edgecolors='none')
ax.scatter(coords[is_healthy,  0], coords[is_healthy,  1],
           c='#2ecc71', alpha=0.8, s=30, label='Healthy',  edgecolors='none')
ax.set_title('t-SNE of SimCLR Learned Representations\n(Healthy vs Diseased leaves)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12, markerscale=2)
ax.axis('off')
plt.tight_layout()
plt.savefig('../outputs/tsne_simclr.png', dpi=150, bbox_inches='tight')
plt.show()